<a href="https://colab.research.google.com/github/zzz-creator/speech-to-text/blob/main/openai_whisper_asr_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Web App Demonstrating OpenAI's Whisper Speech Recognition Model

This is a Colab notebook that allows you to record or upload audio files to [OpenAI's free Whisper speech recognition model](https://openai.com/blog/whisper/). This was based on [an original notebook by @amrrs](https://github.com/amrrs/openai-whisper-webapp), with added documentation and test files by [Pete Warden](https://twitter.com/petewarden).

To use it, choose `Runtime->Run All` from the Colab menu. If you're viewing this notebook on GitHub, follow [this link](https://colab.research.google.com/github/petewarden/openai-whisper-webapp/blob/main/OpenAI_Whisper_ASR_Demo.ipynb) to open it in Colab first. After about a minute or so, you should see a button at the bottom of the page with a `Record from microphone` link. Click this, you'll be asked to give permission to access your mic, and then speak for up to 30 seconds. Once you're done, press `Stop recording`, and a transcript of the first 30 seconds of your speech should soon appear in the box to the right of the recording button. To transcribe more speech, click `Clear' in the left box and start over.

You can also upload your own audio samples using the folder icon on the left of this page. That gives you access to a file system you can upload to by dragging files into it. You can see examples of how to run the transcription in a couple of the cells below.

## Install the Whisper Code

In [ ]:
! pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Load the ML Model

In [ ]:
import whisper

model = whisper.load_model("base")


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 110MiB/s]


## Check we have a GPU

You should see the output `device(type='cuda', index=0)` below. If you don't, you may be on a CPU-only Colab instance which will run more slowly. Go to `Runtime->Change Runtime Type` to fix this.

In [ ]:
model.device

device(type='cuda', index=0)

## Define the Transcribe Function

Now we've loaded the model, and have the code, this is the function that takes an audio file path as an input and returns the recognized text (and logs what it thinks the language is).

In [ ]:
def transcribe(audio):
    # The model's transcribe method handles loading, padding/trimming, and decoding
    # It automatically chunks longer audio files.
    result = model.transcribe(audio)
    print(f"Detected language: {result['language']}")
    return result['text']

## Test with Pre-Recorded Audio

Before we bring up the UI to allow you to record your own live audio, we're going to run the `transcribe()` function on a couple of MP3s we've downloaded. You should see `Mary had a little lamb, its fleece was white as snow, and everywhere that Mary went, the lamb was sure to go.` for `mary.mp3`, which I recorded as an example of clear audio. The second file is a lot harder to transcribe, with very distorted audio, but the model does a good job with `Tazy, Tazy, Tazy. Give me your answer to time after crazy all for the love of you. It won't be a stylish marriage`. You'll notice the transcript is cut off after 30 seconds, which is the default length for this notebook. It can be extended, but that's outside of the scope of this documentation.

In [ ]:
hard_text = transcribe("/content/How_BCSS_Reengineered_High_School_for_Athletes.m4a")
print(hard_text)

Detected language: en
 I want you to just take a second and imagine a public high school where spending your afternoon sleeping or maybe visiting a physical therapist or lifting heavy weights isn't considered skipping class. Right. Where it's actually encouraged. Exactly. In fact, it's legally mandated for your graduation. I really want you to picture the standard high school experience that most of us had. Yeah, the squeaky floors and all that. Oh, exactly. The squeaky linoleum floors, the endless rows of those gray lockers and just the frantic bells that basically dictate your every single move. Moving from an algebra classroom straight to a cramped cafeteria. Yeah. And mixed in with all of that is the inevitable busy work, right? The hours you spend just staring at a clock. Just waiting for the data end. Right. So you can finally go do what you actually care about. But now, imagine a scenario where we just strip all of that rigid architecture away entirely. Which is a huge mental sh

The previous output indicates that the `hard_text` variable contains the full transcription, but the display in the notebook output was truncated for brevity. Let's confirm the length of the `hard_text` and display the first and last few hundred characters to see that the complete text was indeed transcribed.

In [ ]:
print(f"Total length of hard_text: {len(hard_text)} characters")
print("\n--- First 500 characters ---\n")
print(hard_text[:500])
print("\n--- Last 500 characters ---\n")
print(hard_text[-500:])

Total length of hard_text: 26847 characters

--- First 500 characters ---

 I want you to just take a second and imagine a public high school where spending your afternoon sleeping or maybe visiting a physical therapist or lifting heavy weights isn't considered skipping class. Right. Where it's actually encouraged. Exactly. In fact, it's legally mandated for your graduation. I really want you to picture the standard high school experience that most of us had. Yeah, the squeaky floors and all that. Oh, exactly. The squeaky linoleum floors, the endless rows of those gray

--- Last 500 characters ---

a masterclass in layered curriculum design. Let's start with grade 9. This is the foundation. And looking at the schedules, grade 9 is heavily focused on de-streamed academics and establishing baseline biomechanics. They aren't specializing too early. Right. They are building the engine. Yeah. Let's look at Enzo Lau. Enzo is a competitive swimmer. The sources note he swims with the Apex swim

Let's re-run the transcription and inspect the full `result` object returned by the Whisper model to confirm that all segments of the 1-hour audio are being processed.

In [ ]:
print("Running full transcription and inspecting results...")
full_result = model.transcribe("/content/How_BCSS_Reengineered_High_School_for_Athletes.m4a")

print(f"Detected language: {full_result['language']}")
print(f"Total number of segments: {len(full_result['segments'])}")

if len(full_result['segments']) > 0:
    print("\n--- First segment text ---\n")
    print(full_result['segments'][0]['text'])
    print("\n--- Last segment text ---\n")
    print(full_result['segments'][-1]['text'])

# Assign the full concatenated text to hard_text
hard_text = full_result['text']

print(f"\nFull concatenated text length (hard_text): {len(hard_text)} characters")
print("\n--- First 500 characters of hard_text ---\n")
print(hard_text[:500])
print("\n--- Last 500 characters of hard_text ---\n")
print(hard_text[-500:])

Running full transcription and inspecting results...
Detected language: en
Total number of segments: 1047

--- First segment text ---

 I want you to just take a second and imagine a public high school where spending your afternoon

--- Last segment text ---

 build on top of it.

Full concatenated text length (hard_text): 55477 characters

--- First 500 characters of hard_text ---

 I want you to just take a second and imagine a public high school where spending your afternoon sleeping or maybe visiting a physical therapist or lifting heavy weights isn't considered skipping class. Right. Where it's actually encouraged. Exactly. In fact, it's legally mandated for your graduation. I really want you to picture the standard high school experience that most of us had. Yeah, the squeaky floors and all that. Oh, exactly. The squeaky linoleum floors, the endless rows of those gray

--- Last 500 characters of hard_text ---

nes? That's a fascinating question. What if we built a school with thi

The previous output confirms that the `hard_text` variable now holds the full transcription of the 1-hour audio. To easily access and review the entire content, let's save the full text to a `.txt` file. You can then download this file from the Colab file browser (the folder icon on the left sidebar) to view it completely.

In [ ]:
output_filename = 'full_transcription_55k_chars.txt'
with open(output_filename, 'w') as f:
    f.write(hard_text)
print(f"Full transcription saved to {output_filename}")
print("You can download this file from the Colab file browser (left sidebar -> folder icon).")

Full transcription saved to full_transcription_55k_chars.txt
You can download this file from the Colab file browser (left sidebar -> folder icon).
